In [3]:
# =========================
# 0. Install dependencies
# =========================

!pip install -q transformers sentencepiece protobuf tiktoken scikit-learn pandas tqdm torch


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

In [5]:
REPO_URL = input("Enter repository URL: ").strip()
REPO_DIR = Path("/content/AIR_repo")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

Cloning into 'AIR_Group_Task'...
remote: Enumerating objects: 79, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 79 (delta 31), reused 20 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (79/79), 24.38 MiB | 7.15 MiB/s, done.
Resolving deltas: 100% (31/31), done.
Updating files: 100% (26/26), done.
/content/AIR_Group_Task


In [6]:
# =========================
# 1. Imports
# =========================

import os
import re
import json
import sys
import shutil
import subprocess
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
)

In [7]:
# =========================
# 2. Paths and config
# =========================

# repository root
REPO_ROOT = Path.cwd()

while not (
    (REPO_ROOT / "task2" / "reasoning_trace_build.py").exists()
    and (REPO_ROOT / "task2" / "scorer.py").exists()
):
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError(
            "Could not find repo root. Run this notebook from inside the cloned repository."
        )
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)

# drive root for large files
DRIVE_ROOT = Path("/content/drive/MyDrive/AIR_Group_Task")

RAW_TRAIN_PATH = DRIVE_ROOT / "data" / "english" / "english_train.json"
VAL_PATH = DRIVE_ROOT / "data" / "english" / "clef2026_gpt4_o_mini_val.json"

TRAIN_JSONL = DRIVE_ROOT / "output" / "training_data_for_RM" / "english_train.jsonl"
MODEL_DIR = DRIVE_ROOT / "output" / "baseline_roberta"
PRED_PATH = DRIVE_ROOT / "output" / "RM_prediction" / "roberta_predictions.json"
RESULT_DIR = DRIVE_ROOT / "output" / "results_roberta"

BASE_MODEL = "roberta-base"
MAX_LENGTH = 128
BATCH_SIZE = 4
EPOCHS = 3
LR = 1e-5
RANDOM_STATE = 42

os.makedirs(TRAIN_JSONL.parent, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PRED_PATH.parent, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Drive root:", DRIVE_ROOT)
print("Train file exists:", RAW_TRAIN_PATH.exists())
print("Val file exists:", VAL_PATH.exists())
print("Scorer exists:", (REPO_ROOT / "task2" / "scorer.py").exists())


Current directory: /content/AIR_Group_Task
Device: cuda
Train file exists: True
Val file exists: True
Evidence preprocessing exists: True


In [8]:
# =========================
# 3. Run preprocessing
# =========================
subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "task2" / "reasoning_trace_build.py"),
        "--input",
        str(RAW_TRAIN_PATH),
        "--output",
        str(TRAIN_JSONL),
    ],
    check=True,
)




CompletedProcess(args=['/usr/bin/python3', 'task2/reasoning_trace_build.py', '--input', '/content/drive/MyDrive/AIR_CheckThat/data/english/english_train.json', '--output', '/content/drive/MyDrive/AIR_CheckThat/output/training_data_for_RM/english_train.jsonl'], returncode=0)

In [9]:
train_df = pd.read_json(TRAIN_JSONL, lines=True)

print(train_df["input_text"].iloc[0][:1500])
print(train_df.head())
print(train_df.columns)
print(train_df["Class"].value_counts())

train_df["model_input_text"] = train_df["input_text"]

Claim: “The first randomized controlled trial of more than 6,000 individuals to assess the effectiveness of surgical face masks against SARS-CoV-2 infection found masks did not statistically significantly reduce the incidence of infection."
Verdict: false
Justification: To fact check the claim, we need to analyze the statement: “The first randomized controlled trial of more than 6,000 individuals to assess the effectiveness of surgical face masks against SARS-CoV-2 infection found masks did not statistically significantly reduce the incidence of infection."   1. **Numerical Span Analysis**: The claim references "more than 6,000 individuals" which is a specific study size. We need to consider if this number aligns with any evidence provided.  2. **Evidence Review**:    - The evidences state that face masks have at least two effects: preventing viral transmission and protecting non-infected individuals.    - It mentions that surgical masking can correspond to a "65% to 75% reduction in t

In [10]:
# =========================
# 4. Helper functions
# =========================

def remove_label_pattern(text):
    text = re.sub(
        r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting|Supports|Refutes))",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()
    return text.replace("\n", " ")


def build_input(claim, verdict, justification):
    return (
        f"Claim: {claim}\n"
        f"Verdict: {verdict}\n"
        f"Justification: {justification}"
    )


def print_trainable_parameters(model):
    trainable_params = 0
    all_params = 0

    for _, param in model.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    print(
        f"trainable params: {trainable_params} || "
        f"all params: {all_params} || "
        f"trainable%: {100 * trainable_params / all_params:.2f}"
    )


In [11]:
# =========================
# 5. Dataset
# =========================

class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["model_input_text"].tolist()
        self.labels = dataframe["Class"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)

        return item


In [12]:
# =========================
# 6. RoBERTa verifier model
# =========================

class CustomClassifier(torch.nn.Module):
    def __init__(
            self,
            model_name,
            num_labels=1,
            hidden_dim=None,
            dropout_value=0.1,
            freeze_base_layer=False,
    ):
        super().__init__()

        self.model = AutoModel.from_pretrained(model_name)

        if freeze_base_layer:
            for param in self.model.parameters():
                param.requires_grad = False

        hidden_size = self.model.config.hidden_size

        if hidden_dim:
            self.classifier = torch.nn.Sequential(
                torch.nn.Linear(hidden_size, hidden_dim),
                torch.nn.ReLU(),
                torch.nn.Dropout(dropout_value),
                torch.nn.Linear(hidden_dim, num_labels),
            )
        else:
            self.classifier = torch.nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = pooled_output.float()

        logits = self.classifier(pooled_output)
        return logits

In [13]:
# =========================
# 7. Trainer
# =========================

class TrainerModule:
    def __init__(
            self,
            model,
            train_loader,
            val_loader,
            epochs,
            lr,
            output_dir,
    ):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)

        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs = epochs

        self.optimizer = AdamW(self.model.parameters(), lr=lr, eps=1e-8)
        self.loss_fn = torch.nn.BCEWithLogitsLoss()

        total_steps = len(train_loader) * epochs
        warmup_steps = int(0.05 * total_steps)

        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            warmup_steps,
            total_steps,
        )

        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

    def train(self):
        for epoch in range(self.epochs):
            print(f"\nEpoch {epoch + 1}/{self.epochs}")
            self.model.train()

            total_loss = 0
            total_acc = 0

            for batch in tqdm(self.train_loader):
                self.optimizer.zero_grad()

                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                logits = self.model(input_ids, attention_mask).squeeze(-1)
                logits = torch.clamp(logits, -50, 50)
                labels = labels.float()
                loss = self.loss_fn(logits, labels)


                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()
                self.scheduler.step()

                total_loss += loss.item()

                preds = (
                        torch.sigmoid(logits) >= 0.5
                ).detach().cpu().numpy()

                total_acc += accuracy_score(
                    labels.detach().cpu().numpy(),
                    preds,
                )

            print(f"Train Loss: {total_loss / len(self.train_loader):.4f}")
            print(f"Train Acc: {total_acc / len(self.train_loader):.4f}")

            self.evaluate(epoch)

    def evaluate(self, epoch):
        self.model.eval()

        total_loss = 0
        total_acc = 0

        with torch.no_grad():
            for batch in self.val_loader:
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                logits = self.model(input_ids, attention_mask).squeeze(-1)
                logits = torch.clamp(logits, -50, 50)
                labels = labels.float()
                loss = self.loss_fn(logits, labels)


                total_loss += loss.item()

                preds = (
                        torch.sigmoid(logits) >= 0.5
                ).detach().cpu().numpy()

                total_acc += accuracy_score(
                    labels.detach().cpu().numpy(),
                    preds,
                )

        print(f"Val Loss: {total_loss / len(self.val_loader):.4f}")
        print(f"Val Acc: {total_acc / len(self.val_loader):.4f}")

        tokenizer.save_pretrained(self.output_dir)

        torch.save(
            self.model.state_dict(),
            self.output_dir / f"model_epoch_{epoch}.pt",
        )


In [14]:
# =========================
# 8. Train RoBERTa verifier
# =========================

train_split, dev_split = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["Class"],
    random_state=RANDOM_STATE,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

train_dataset = TextDataset(train_split, tokenizer, MAX_LENGTH)
dev_dataset = TextDataset(dev_split, tokenizer, MAX_LENGTH)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
)

model = CustomClassifier(
    model_name=BASE_MODEL,
    freeze_base_layer=False,
)

print_trainable_parameters(model)

trainer = TrainerModule(
    model=model,
    train_loader=train_loader,
    val_loader=dev_loader,
    epochs=EPOCHS,
    lr=LR,
    output_dir=MODEL_DIR,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Train Loss: 0.5838
Train Acc: 0.7843
Val Loss: 0.4654
Val Acc: 0.8212

Epoch 2/3


100%|██████████| 6287/6287 [14:38<00:00,  7.16it/s]


Train Loss: 0.5500
Train Acc: 0.8375
Val Loss: 0.5947
Val Acc: 0.8446

Epoch 3/3


100%|██████████| 6287/6287 [14:32<00:00,  7.20it/s]


Train Loss: 0.4782
Train Acc: 0.8669
Val Loss: 0.6200
Val Acc: 0.8473


In [15]:
# =========================
# 9. Prediction evaluator
# =========================

class VerifierEvaluator:
    def __init__(
            self,
            model_path,
            tokenizer_path,
            base_model,
            device="cuda",
    ):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

        self.model = CustomClassifier(
            model_name=base_model,
            freeze_base_layer=False,
        )

        self.model.load_state_dict(
            torch.load(model_path, map_location=self.device)
        )

        self.model.to(self.device)
        self.model.eval()

    def encode_input(
            self,
            claim,
            verdict,
            justification,
            max_length=MAX_LENGTH,
    ):
        text = build_input(
            claim=claim,
            verdict=verdict,
            justification=justification,
        )

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

        return (
            encoding["input_ids"].to(self.device),
            encoding["attention_mask"].to(self.device),
        )

    def score(self, claim, verdict, justification):
        input_ids, attention_mask = self.encode_input(
            claim=claim,
            verdict=verdict,
            justification=justification,
        )

        with torch.no_grad():
            score = self.model(input_ids, attention_mask).item()

        return float(score)

In [16]:
# =========================
# 10. Generate predictions
# =========================

BEST_EPOCH = EPOCHS - 1
MODEL_PATH = MODEL_DIR / f"model_epoch_{BEST_EPOCH}.pt"

with open(VAL_PATH, "r", encoding="utf-8") as f:
    val_data = json.load(f)

evaluator = VerifierEvaluator(
    model_path=MODEL_PATH,
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
)

predictions = []

for idx, sample in enumerate(tqdm(val_data)):
    claim = sample["claim"]
    verdict_list = []
    verifier_score_list = []
    justification_list = []

    for trace_idx in range(len(sample["Reasoning_traces"])):
        justification = remove_label_pattern(
            sample["Reasoning_traces"][trace_idx]
        ).split("Label:")[0]

        verdict = sample["Verdict_list"][trace_idx].lower()

        score = evaluator.score(
            claim=claim,
            verdict=verdict,
            justification=justification,
        )

        verdict_list.append(sample["Verdict_list"][trace_idx])
        justification_list.append(justification)
        verifier_score_list.append(score)

    best_idx = int(np.argmax(np.array(verifier_score_list)))
    best_verdict = verdict_list[best_idx]

    predictions.append(
        {
            "query_id": sample.get("query_id", idx),
            "Claim": claim,
            "Label": sample["label"],
            "Verdict_BoN": best_verdict,
            "BoN_Verdict_list": verdict_list,
            "Reasoning_traces": justification_list,
            "score_list": verifier_score_list,
        }
    )

with open(PRED_PATH, "w", encoding="utf-8") as fp:
    json.dump(predictions, fp, indent=4, ensure_ascii=False)

print(f"Saved predictions to {PRED_PATH}")
print("Number of predictions:", len(predictions))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 1600/1600 [04:46<00:00,  5.59it/s]


Saved predictions to /content/drive/MyDrive/AIR_CheckThat/output/RM_prediction/roberta_predictions.json
Number of predictions: 1600


In [17]:
print("PRED_PATH exists:", os.path.exists(PRED_PATH))
print("PRED_PATH:", PRED_PATH)

PRED_PATH exists: True
PRED_PATH: /content/drive/MyDrive/AIR_CheckThat/output/RM_prediction/roberta_predictions.json


In [18]:
# =========================
# 11. Run provided scorer
# =========================

REPO_PRED_DIR = REPO_ROOT / "output" / "RM_prediction"
os.makedirs(REPO_PRED_DIR, exist_ok=True)

shutil.copy(
    PRED_PATH,
    REPO_PRED_DIR / "clef_predictions.json",
)

subprocess.run(
    [sys.executable, str(REPO_ROOT / "task2" / "scorer.py")],
    check=True,
    cwd=str(REPO_ROOT),
)

shutil.copy(
    REPO_PRED_DIR / "result.csv",
    RESULT_DIR / "result.csv",
)

shutil.copy(
    REPO_PRED_DIR / "per_sample_ir.csv",
    REPO_PRED_DIR / "per_sample_ir.csv",
    RESULT_DIR / "per_sample_ir.csv",
)

print(f"Result saved to {RESULT_DIR / 'result.csv'}")
print(f"Per-sample IR saved to {RESULT_DIR / 'per_sample_ir.csv'}")

Result saved to /content/drive/MyDrive/AIR_CheckThat/output/results_roberta/result.csv
Per-sample IR saved to /content/drive/MyDrive/AIR_CheckThat/output/results_roberta/per_sample_ir.csv
